# IntentGate Z: evaluation on IBM LinuxONE

**Authorized doesn't always mean intended.** IntentGate Z is a real-time gateway that checks whether an autonomous AI agent's proposed tool call still matches what the human operator authorised, then returns **ALLOW**, **CHALLENGE** or **BLOCK** before anything executes.

This notebook reproduces the project's evidence on the machine it runs on:

1. which hardware and software it is running on (verified, not assumed);
2. how the synthetic emergency-operations dataset is generated;
3. training and held-out evaluation of the text models and the risk model;
4. rules vs ML vs hybrid comparison, per-attack-family detection and ablations;
5. the four headline scenarios through the real gateway, with the audit chain verified;
6. latency/throughput and the CPU vs Telum NNPA inference comparison.

All operational data is synthetic. Labels come from the generator's latent scenario family, not from real incidents.

In [ ]:
import json, os, platform, re, subprocess, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'backend'))
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
cpu = Path('/proc/cpuinfo').read_text() if Path('/proc/cpuinfo').exists() else ''
feats = next((l.split(':', 1)[1].split() for l in cpu.splitlines() if l.startswith('features')), [])
machine = re.search(r'machine\s*=\s*([0-9A-F]{4})', cpu)
print('architecture   :', platform.machine())
print('machine type   :', machine.group(1) if machine else 'n/a', '(9175 = IBM z17)')
print('nnpa facility  :', 'nnpa' in feats)
print('python         :', platform.python_version())
print('cpu count      :', os.cpu_count())
try:
    print(subprocess.run(['lscpu'], capture_output=True, text=True).stdout)
except Exception as e:
    print('lscpu unavailable:', e)

In [ ]:
from importlib import metadata
for p in ['numpy', 'pandas', 'scikit-learn', 'xgboost', 'tensorflow', 'ibm-zdnn-plugin', 'torch', 'fastapi', 'pydantic']:
    try:
        print(f'{p:16s}', metadata.version(p))
    except metadata.PackageNotFoundError:
        print(f'{p:16s}', 'not installed')

## 1. Text models (instruction-injection signals)

Two TF-IDF + logistic-regression models, each used only on its own channel:

* **feed model**: does untrusted feed/tool text try to instruct the agent?
* **rationale model**: does the agent's own rationale defer to an outside notice instead of the operator?

Template variants are split train/val/test so the test set uses unseen phrasings. The feed model is also scored on public attack data (InjecAgent and AgentDojo, MIT licence) that it never trained on.

In [ ]:
r = subprocess.run([sys.executable, str(ROOT/'ml/train_text.py')], capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-3000:] or r.stderr[-3000:])

## 2. Synthetic emergency-operations dataset

Every row is produced by running the real gateway code: a randomised city state, a signed human intent, an agent proposal, the hard policy engine and the feature extractor. 27 scenario families (benign, legitimately exceptional, and adversarial) with leakage controls: split-disjoint text templates, partially-tagged provenance, benign texts containing suspicious words, polite injections without override words, and 1-2% label noise between adjacent classes.

In [ ]:
r = subprocess.run([sys.executable, str(ROOT/'ml/generate_dataset.py'), '--n', '12000', '--seed', '7', '--text-model', str(ROOT/'ml/artifacts')], capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-2000:] or r.stderr[-2000:])
import pandas as pd
df = pd.read_csv(ROOT/'datasets/intentgate_synthetic_seed7.csv')
display(df.groupby(['split', 'label']).size().unstack())
display(df[df.split == 'test'].family.value_counts().head(30))

## 3. Risk models: train, select on validation, evaluate once on test

Candidates: logistic regression, random forest, histogram gradient boosting, MLP, XGBoost (when installed) and a Keras MLP (for the TensorFlow / NNPA path). Thresholds are chosen on validation subject to a false-block rate of at most 2% on legitimate actions; the serving model is selected by validation macro-F1 of the hybrid gateway.

In [ ]:
r = subprocess.run([sys.executable, str(ROOT/'ml/train.py'), '--data', str(ROOT/'datasets/intentgate_synthetic_seed7.csv')], capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-4000:] or r.stderr[-4000:])

In [ ]:
res = json.loads((ROOT/'ml/results/evaluation_seed7.json').read_text())
rows = []
for k, name in [('hard_policy', 'Hard policy only'), ('rules_heuristic', 'Rules + heuristics'), ('ml_only', 'ML only'), ('hybrid', 'IntentGate Z hybrid')]:
    m = res['test'][k]
    rows.append({'method': name, 'macro_f1': m['macro_f1'], 'accuracy': m['accuracy'], 'unsafe_miss_rate': m['unsafe_miss_rate'], 'false_block_rate_allow': m['false_block_rate_allow'], 'legit_acceptance': m['legit_acceptance_rate'], 'challenge_rate': m['challenge_rate']})
print('selected model:', res['selected_model'], '| thresholds:', res['thresholds'], '| trained on', res['platform']['architecture'])
pd.DataFrame(rows)

In [ ]:
from IPython.display import Image, display
for c in res.get('charts', []):
    print(c)
    display(Image(filename=str(ROOT/'ml/results/charts'/f'{c}.png')))

## 4. The four headline scenarios through the real gateway

In [ ]:
os.environ['INTENTGATE_DB_PATH'] = ':memory:'
from fastapi.testclient import TestClient
from intentgate.api import app
with TestClient(app) as c:
    s = c.post('/api/session').json(); H = {'X-IG-Session': s['session_id']}
    for sc in ['reroute', 'supply', 'injection', 'drift']:
        for d in c.post(f'/api/scenarios/{sc}/run', headers=H).json()['decisions']:
            flags = [f['code'] for f in d['findings'] if f['outcome'] in ('block', 'challenge')]
            print(f"{sc:10s} {d['verdict']:9s} {d['status']:15s} decision {d['latency_ms']['decision_ms']:.3f} ms  risk={d['risk']['risk_score'] if d['risk'] else None}  {flags}")
    print(c.get('/api/audit/verify').json())
    print(c.get('/api/model/info').json()['risk_model']['runtime'])

## 5. Performance: gateway latency and CPU vs NNPA inference

In [ ]:
r = subprocess.run([sys.executable, str(ROOT/'ml/benchmark.py'), '--mode', 'inprocess', '--n', '2000', '--reps', '3'], capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-2000:] or r.stderr[-2000:])

In [ ]:
r = subprocess.run([sys.executable, str(ROOT/'ml/benchmark_nnpa.py'), '--reps', '30'], capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-3000:] or r.stderr[-3000:])

## Limitations

* Synthetic data: the generator encodes our assumptions about emergency operations; scores measure consistency with those assumptions, not real-world accuracy.
* Public attack sets target consumer tools, not emergency operations; they test transfer of the text signal only.
* NNPA results apply to the TensorFlow MLP path only and only when the process lists an NNPA device. The serving model may run on CPU if that is faster for single requests; the API reports which runtime is live.
* The audit chain detects tampering; it does not make the log immutable.